# Session 09 — Object Detection

**CVI4IC · Summer Semester 2026 · FH Upper Austria**

> Today we move from *“what is in this image?”* to *“what is in this image, and where?”*
> We work with three complementary detectors:
>
> 1. **YOLO11** — Ultralytics, the anchor-free one-stage workhorse.
> 2. **RT-DETR** — Baidu's real-time DETR; no NMS, transformer queries.
> 3. **Vision-Language Models** — PaliGemma (boxes as `<loc>` tokens) and Gemma 3 (boxes as JSON): the LM writes the boxes out as text.
>
> All three are exercised on the same set of images so you can compare.

> **Colab tip:** Runtime → Change runtime type → **GPU** before running. We use a small T4 happily.


## 0. Setup

The first cell installs the three libraries we need that aren't already in Colab:
* `ultralytics` — YOLO11 + auxiliary trainers.
* `transformers` — HuggingFace, for RT-DETR, PaliGemma, and Gemma 3.
* `supervision` — clean drawing utilities (boxes, labels, NMS).


In [ ]:
!pip install -q ultralytics transformers accelerate supervision pillow

In [ ]:
import os, io, time, math, urllib.request, zipfile, random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "·  torch", torch.__version__)
random.seed(42); np.random.seed(42); torch.manual_seed(42)
plt.rcParams["figure.dpi"] = 110

## 1. Sample images

For detection demos we use a handful of real images with multiple objects. Fruits-360 is single-object so we additionally pull a couple of multi-object scenes from public sources.


In [ ]:
SAMPLE_URLS = {
    "kitchen.jpg": "https://images.unsplash.com/photo-1546069901-ba9599a7e63c?w=900",  # bowl of fruit
    "market.jpg":  "https://images.unsplash.com/photo-1488459716781-31db52582fe9?w=900",  # market stall
    "dog_bike.jpg":"https://ultralytics.com/images/bus.jpg",  # canonical YOLO test image (bus + people)
}

DATA = Path("data09"); DATA.mkdir(exist_ok=True)

for fname, url in SAMPLE_URLS.items():
    p = DATA / fname
    if not p.exists():
        try:
            urllib.request.urlretrieve(url, p)
            print(f"Saved {fname}")
        except Exception as e:
            print(f"Couldn't download {fname}: {e}")

# Show what we have
imgs = [Image.open(p).convert("RGB") for p in DATA.glob("*.jpg") if p.is_file()]
fig, axes = plt.subplots(1, len(imgs), figsize=(4*len(imgs), 4))
if len(imgs) == 1: axes = [axes]
for ax, img, p in zip(axes, imgs, sorted(DATA.glob("*.jpg"))):
    ax.imshow(img); ax.set_title(p.name); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Part A — YOLO11 with Ultralytics

The Ultralytics package wraps detector training, inference, and visualization. A pretrained `yolo11n.pt` model is ~6 MB and ships with COCO weights (80 classes).


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")           # nano variant — small + fast
print(model.info(verbose=False))
print("Predicting on", DATA / "dog_bike.jpg")
res = model.predict(source=str(DATA / "dog_bike.jpg"), conf=0.25, verbose=False)[0]
print("Detections:", len(res.boxes))

### Look at the predictions

In [ ]:
def show_boxes_pil(image_path, boxes, labels=None, scores=None, color="lime", ax=None):
    """Draw [xyxy] boxes on an image with optional labels."""
    img = Image.open(image_path).convert("RGB")
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img); ax.axis("off")
    W, H = img.size
    for i, b in enumerate(boxes):
        x1, y1, x2, y2 = b
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, linewidth=2, edgecolor=color)
        ax.add_patch(rect)
        if labels is not None:
            txt = labels[i]
            if scores is not None: txt += f" {scores[i]:.2f}"
            ax.text(x1, max(0, y1-4), txt, color="black",
                    fontsize=9, bbox=dict(facecolor=color, alpha=0.7, edgecolor="none", pad=1))
    return ax

# Extract YOLO predictions
boxes  = res.boxes.xyxy.cpu().numpy()
clsids = res.boxes.cls.int().cpu().tolist()
confs  = res.boxes.conf.cpu().tolist()
names  = [res.names[i] for i in clsids]

ax = show_boxes_pil(DATA / "dog_bike.jpg", boxes, names, confs, color="lime")
ax.set_title("YOLO11n detections (conf ≥ 0.25)")
plt.show()

### NMS in action — before vs after

YOLO does NMS internally. To see *why* it matters, we ask the model to emit all per-anchor predictions before NMS (`agnostic_nms=False`, high IoU threshold) and compare to the cleaned output.


In [ ]:
# Get RAW predictions with a permissive NMS (lots of overlapping survivors)
res_raw = model.predict(source=str(DATA / "dog_bike.jpg"), conf=0.05, iou=0.95, verbose=False)[0]

# Standard (default) predictions — NMS already applied
res_nms = model.predict(source=str(DATA / "dog_bike.jpg"), conf=0.25, iou=0.5, verbose=False)[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
show_boxes_pil(DATA / "dog_bike.jpg",
               res_raw.boxes.xyxy.cpu().numpy(),
               labels=[res_raw.names[int(c)] for c in res_raw.boxes.cls],
               scores=res_raw.boxes.conf.cpu().tolist(),
               color="red", ax=axes[0])
axes[0].set_title(f"Before NMS — {len(res_raw.boxes)} boxes")

show_boxes_pil(DATA / "dog_bike.jpg",
               res_nms.boxes.xyxy.cpu().numpy(),
               labels=[res_nms.names[int(c)] for c in res_nms.boxes.cls],
               scores=res_nms.boxes.conf.cpu().tolist(),
               color="lime", ax=axes[1])
axes[1].set_title(f"After NMS — {len(res_nms.boxes)} boxes")
plt.tight_layout(); plt.show()

### How fast is it?

YOLO11n on a T4 should comfortably hit > 100 fps. Let's measure.


In [ ]:
# warm-up + timing
for _ in range(3): model.predict(source=str(DATA / "dog_bike.jpg"), verbose=False)

t0 = time.time(); N = 30
for _ in range(N):
    _ = model.predict(source=str(DATA / "dog_bike.jpg"), verbose=False)
dt = (time.time() - t0) / N
print(f"YOLO11n: {1/dt:.1f} fps  ({dt*1000:.1f} ms / image)")

### Fine-tuning YOLO on a tiny custom dataset

To show how easy training is, we build a one-class detector for **apples** using a handful of Fruits-360 single-object images, synthetically pasted on random backgrounds.

> In practice you would use a curated bounding-box dataset (COCO / VOC format). Ultralytics expects either a YOLO-format folder or a small `dataset.yaml`.


In [ ]:
!git clone --depth 1 https://github.com/fruits-360/fruits-360-100x100.git 2>/dev/null
APPLE_DIR = Path("fruits-360-100x100/Training/Apple Braeburn 1")
print("Apple images available:", len(list(APPLE_DIR.glob("*.jpg"))))

In [ ]:
# Build a tiny synthetic detection dataset by pasting apples onto random backgrounds.
SYN = Path("yolo_apples"); SYN.mkdir(exist_ok=True)
(SYN / "images" / "train").mkdir(parents=True, exist_ok=True)
(SYN / "images" / "val").mkdir(parents=True, exist_ok=True)
(SYN / "labels" / "train").mkdir(parents=True, exist_ok=True)
(SYN / "labels" / "val").mkdir(parents=True, exist_ok=True)

apple_paths = sorted(APPLE_DIR.glob("*.jpg"))[:50]
random.shuffle(apple_paths)
train_paths = apple_paths[:35]; val_paths = apple_paths[35:]

def make_sample(apple_p, out_img, out_label, canvas_size=320, n_apples=(1,3)):
    canvas = Image.new("RGB", (canvas_size, canvas_size),
                       color=tuple(random.randint(50, 200) for _ in range(3)))
    # add some noise tiles for texture
    for _ in range(12):
        x = random.randint(0, canvas_size-20); y = random.randint(0, canvas_size-20)
        canvas.paste(tuple(random.randint(0, 255) for _ in range(3)), (x, y, x+20, y+20))
    n = random.randint(*n_apples)
    rows = []
    for _ in range(n):
        apple = Image.open(apple_p).convert("RGB")
        s = random.randint(60, 110)
        apple = apple.resize((s, s))
        # 2D translation, avoid clipping
        x = random.randint(0, canvas_size - s); y = random.randint(0, canvas_size - s)
        canvas.paste(apple, (x, y))
        # YOLO label: cls cx cy w h  (all normalised)
        cx = (x + s/2)/canvas_size; cy = (y + s/2)/canvas_size
        rows.append(f"0 {cx:.4f} {cy:.4f} {s/canvas_size:.4f} {s/canvas_size:.4f}")
    canvas.save(out_img)
    out_label.write_text("\n".join(rows))

for i, p in enumerate(train_paths):
    make_sample(p, SYN / "images/train" / f"t{i:03d}.jpg", SYN / "labels/train" / f"t{i:03d}.txt")
for i, p in enumerate(val_paths):
    make_sample(p, SYN / "images/val"   / f"v{i:03d}.jpg", SYN / "labels/val"   / f"v{i:03d}.txt")

YAML = SYN / "dataset.yaml"
YAML.write_text(f"""path: {SYN.resolve()}
train: images/train
val:   images/val
names:
  0: apple
""")
print("Wrote dataset.yaml at", YAML)

In [ ]:
# Quick look at one synthetic sample with its label
sample_img = sorted((SYN / "images/train").glob("*.jpg"))[0]
sample_lbl = (SYN / "labels/train" / (sample_img.stem + ".txt")).read_text().strip().splitlines()
img = Image.open(sample_img); W, H = img.size
boxes_xyxy = []
for line in sample_lbl:
    _, cx, cy, w, h = map(float, line.split())
    x1 = (cx - w/2) * W; y1 = (cy - h/2) * H
    x2 = (cx + w/2) * W; y2 = (cy + h/2) * H
    boxes_xyxy.append((x1, y1, x2, y2))

ax = show_boxes_pil(sample_img, boxes_xyxy, ["apple"]*len(boxes_xyxy), color="orange")
ax.set_title(f"Synthetic training sample ({len(boxes_xyxy)} apples)")
plt.show()

In [ ]:
# Fine-tune YOLO11n for 5 epochs on the tiny synthetic apple set.
# This runs in ~30 s on a T4. Tweak imgsz / epochs as needed.
model_ft = YOLO("yolo11n.pt")
model_ft.train(data=str(YAML), epochs=5, imgsz=320,
               batch=16, plots=False, verbose=False, exist_ok=True)
metrics = model_ft.val(verbose=False)
print(f"\nApple-detector mAP@50  = {metrics.box.map50:.3f}")
print(f"Apple-detector mAP50-95 = {metrics.box.map:.3f}")

In [ ]:
# Predict on a few held-out validation samples
val_imgs = sorted((SYN / "images/val").glob("*.jpg"))[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, p in zip(axes, val_imgs):
    r = model_ft.predict(source=str(p), conf=0.25, verbose=False)[0]
    show_boxes_pil(p, r.boxes.xyxy.cpu().numpy(),
                   ["apple"]*len(r.boxes), r.boxes.conf.cpu().tolist(),
                   color="lime", ax=ax)
    ax.set_title(p.name)
plt.tight_layout(); plt.show()

## 3. Part B — RT-DETR via HuggingFace

RT-DETR (Baidu, 2024) is a DETR variant designed for real-time use. It removes NMS, uses a hybrid encoder, and reaches YOLO-class speed.

We use the `PekingU/rtdetr_v2_r18vd` checkpoint — the small student model.


In [ ]:
from transformers import RTDetrImageProcessor, RTDetrForObjectDetection

rt_id = "PekingU/rtdetr_v2_r18vd"
rt_proc  = RTDetrImageProcessor.from_pretrained(rt_id)
rt_model = RTDetrForObjectDetection.from_pretrained(rt_id).to(device).eval()
print(f"RT-DETR loaded ({sum(p.numel() for p in rt_model.parameters())/1e6:.1f} M params)")

In [ ]:
def rtdetr_predict(image_path, threshold=0.3):
    img = Image.open(image_path).convert("RGB")
    inputs = rt_proc(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = rt_model(**inputs)
    target = torch.tensor([img.size[::-1]]).to(device)  # (H, W)
    res = rt_proc.post_process_object_detection(outputs, target_sizes=target, threshold=threshold)[0]
    boxes  = res["boxes"].cpu().numpy()
    scores = res["scores"].cpu().tolist()
    labels = [rt_model.config.id2label[i.item()] for i in res["labels"]]
    return boxes, labels, scores

boxes, names, confs = rtdetr_predict(DATA / "dog_bike.jpg", threshold=0.3)
print(f"RT-DETR found {len(boxes)} objects")
ax = show_boxes_pil(DATA / "dog_bike.jpg", boxes, names, confs, color="cyan")
ax.set_title("RT-DETR predictions"); plt.show()

### Side-by-side: YOLO11 vs RT-DETR

Same images, both detectors. Look for where they agree and where they disagree.


In [ ]:
def yolo_predict(image_path, threshold=0.25):
    res = model.predict(source=str(image_path), conf=threshold, verbose=False)[0]
    boxes  = res.boxes.xyxy.cpu().numpy()
    names  = [res.names[int(c)] for c in res.boxes.cls]
    confs  = res.boxes.conf.cpu().tolist()
    return boxes, names, confs

paths = [DATA / "dog_bike.jpg", DATA / "kitchen.jpg", DATA / "market.jpg"]
paths = [p for p in paths if p.exists()]
fig, axes = plt.subplots(len(paths), 2, figsize=(14, 5*len(paths)))
if len(paths) == 1: axes = axes[None, :]
for row, p in enumerate(paths):
    yb, yl, yc = yolo_predict(p, 0.25)
    rb, rl, rc = rtdetr_predict(p, 0.3)
    show_boxes_pil(p, yb, yl, yc, color="lime", ax=axes[row][0])
    axes[row][0].set_title(f"YOLO11n — {p.name}  ({len(yb)} boxes)")
    show_boxes_pil(p, rb, rl, rc, color="cyan", ax=axes[row][1])
    axes[row][1].set_title(f"RT-DETR — {p.name}  ({len(rb)} boxes)")
plt.tight_layout(); plt.show()

### Speed comparison


In [ ]:
# Speed comparison (single image, T4 GPU)
sample = DATA / "dog_bike.jpg"

# YOLO warmup + timing
for _ in range(3): yolo_predict(sample)
t0 = time.time(); N = 30
for _ in range(N): yolo_predict(sample)
yolo_ms = (time.time() - t0) / N * 1000

# RT-DETR warmup + timing
for _ in range(3): rtdetr_predict(sample)
t0 = time.time();
for _ in range(N): rtdetr_predict(sample)
rt_ms = (time.time() - t0) / N * 1000

print(f"YOLO11n   : {yolo_ms:5.1f} ms / image  ({1000/yolo_ms:.0f} fps)")
print(f"RT-DETR v2 r18: {rt_ms:5.1f} ms / image  ({1000/rt_ms:.0f} fps)")

## 4. Part C — VLM detection: PaliGemma → Gemma

We end with the third paradigm: let a language model **write the boxes**. We do it twice, the way the field evolved:

* **C.1 — PaliGemma** (Google, 2024): the model that *started* grounded VLM output. Boxes are special `<loc0000>` … `<loc1023>` tokens.
* **C.2 — Gemma 3** (Google, 2025): a *general* multimodal model. Same idea, but it emits clean **JSON** you can parse with the standard library.

> **Heads up:** both are gated HF repos. Run `huggingface-cli login` once with an HF token (or set `HF_TOKEN`). If you hit a 401, the fallbacks below use the older non-gated checkpoints.


### C.1 — PaliGemma: boxes as `<loc>` tokens

In [ ]:
# Optional: log in for gated repos (uncomment if you have a HF token).
# from huggingface_hub import login
# login(token=os.environ.get("HF_TOKEN", ""))

In [ ]:
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor

pali_id = "google/paligemma2-3b-mix-224"  # gated. Fallback: "google/paligemma-3b-mix-224"
try:
    pali_proc = PaliGemmaProcessor.from_pretrained(pali_id)
    pali = PaliGemmaForConditionalGeneration.from_pretrained(
        pali_id, torch_dtype=torch.float16, device_map=device).eval()
    print(f"Loaded {pali_id}")
except Exception as e:
    print("Couldn't load gated PaliGemma 2:", e)
    print("Falling back to PaliGemma 1 (non-gated).")
    pali_id = "google/paligemma-3b-mix-224"
    pali_proc = PaliGemmaProcessor.from_pretrained(pali_id)
    pali = PaliGemmaForConditionalGeneration.from_pretrained(
        pali_id, torch_dtype=torch.float16, device_map=device).eval()

**Parsing `<locXXXX>` tokens.** PaliGemma emits `<loc0273><loc0451><loc0879><loc0935> apple` per detection.
Order: **y_min, x_min, y_max, x_max**, each ∈ [0, 1024) normalised; followed by the class label. The helper below decodes it back to pixel boxes.

In [ ]:
import re

def parse_paligemma_detect(text, img_size):
    """Convert PaliGemma text → [(label, [x1,y1,x2,y2])] in pixel coords."""
    W, H = img_size
    out = []
    tokens = list(re.finditer(r"<loc(\d{4})>", text))
    i = 0
    while i + 4 <= len(tokens):
        nums = [int(tokens[i+k].group(1)) for k in range(4)]
        end = tokens[i+3].end()
        nxt = tokens[i+4].start() if i+4 < len(tokens) else len(text)
        label = text[end:nxt].strip(" ;,\n")
        y1 = nums[0] / 1024 * H        # PaliGemma order: y_min, x_min, y_max, x_max
        x1 = nums[1] / 1024 * W
        y2 = nums[2] / 1024 * H
        x2 = nums[3] / 1024 * W
        out.append((label, [x1, y1, x2, y2]))
        i += 4
    return out

In [ ]:
def pali_detect(image_path, prompt="detect apple ; banana ; cucumber ; fruit", max_new_tokens=64):
    img = Image.open(image_path).convert("RGB")
    inputs = pali_proc(text=prompt, images=img, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = pali.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[0][inputs["input_ids"].shape[-1]:]
    text = pali_proc.decode(gen, skip_special_tokens=False)
    return text, parse_paligemma_detect(text, img.size)

target_img = DATA / "kitchen.jpg" if (DATA / "kitchen.jpg").exists() else DATA / "dog_bike.jpg"
prompt = "detect fruit ; bowl ; banana ; apple"
raw, dets = pali_detect(target_img, prompt=prompt)
print("Raw PaliGemma output:")
print(repr(raw))
print(f"\nParsed {len(dets)} detections.")

In [ ]:
# Visualise the PaliGemma detections
if dets:
    boxes = [b for _, b in dets]; labels = [l for l, _ in dets]
    ax = show_boxes_pil(target_img, boxes, labels, color="magenta")
    ax.set_title(f"PaliGemma open-vocabulary detection — {prompt!r}")
    plt.show()
else:
    print("No <loc> tokens emitted. Try a different prompt or image.")

**Referring expressions — *find me this thing*.** Where VLMs shine: detect the *thing* you described, even if the class name is unusual.

In [ ]:
ref_prompts = [
    "detect the leftmost object",
    "detect the round red fruit",
    "detect the largest item in the scene",
]
fig, axes = plt.subplots(1, len(ref_prompts), figsize=(5*len(ref_prompts), 5))
for ax, p in zip(axes, ref_prompts):
    raw, dets = pali_detect(target_img, prompt=p, max_new_tokens=32)
    if dets:
        boxes = [b for _, b in dets]; labels = [l for l, _ in dets]
        show_boxes_pil(target_img, boxes, labels, color="magenta", ax=ax)
    else:
        ax.imshow(Image.open(target_img).convert("RGB")); ax.axis("off")
    ax.set_title(f"{p}\n{(dets[0][0] if dets else '— no detection —')[:30]}", fontsize=10)
plt.tight_layout(); plt.show()

### C.2 — Gemma 3: boxes as JSON

PaliGemma's `<loc>` tokens need a custom decoder. The newer, *general-purpose* **Gemma 3** simply returns **JSON** when you ask for it — coordinates normalised to **0–1000** (note: 1000, not 1024). No special tokens, no custom parser: just `json.loads`.

In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

gemma_id = "google/gemma-3-4b-it"   # gated; request access on the HF model page
gproc  = AutoProcessor.from_pretrained(gemma_id)
gmodel = Gemma3ForConditionalGeneration.from_pretrained(
    gemma_id, torch_dtype=torch.bfloat16, device_map=device).eval()
print(f"Loaded {gemma_id}")

In [ ]:
import json

def parse_gemma_json(text, img_size):
    """Parse Gemma's JSON detections → [(label, [x1,y1,x2,y2])] in pixel coords."""
    W, H = img_size
    m = re.search(r"\[.*\]", text, re.DOTALL)          # grab the JSON list, ignore prose / ``` fences
    if not m:
        return []
    try:
        data = json.loads(m.group(0))
    except json.JSONDecodeError:
        return []
    out = []
    for d in data:
        if "box_2d" not in d:
            continue
        ymin, xmin, ymax, xmax = d["box_2d"]            # Gemma order: y_min, x_min, y_max, x_max in 0-1000
        box = [xmin/1000*W, ymin/1000*H, xmax/1000*W, ymax/1000*H]
        out.append((d.get("label", "?"), box))
    return out

In [ ]:
def gemma_detect(image_path, classes="apple, banana, cucumber, fruit", max_new_tokens=256):
    img = Image.open(image_path).convert("RGB")
    prompt = (f"Detect these objects: {classes}. "
              'Return ONLY a JSON list; each item is '
              '{"box_2d": [ymin, xmin, ymax, xmax], "label": <name>} '
              "with coordinates normalised to 0-1000.")
    messages = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text",  "text": prompt}]}]
    inputs = gproc.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(device)
    in_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = gmodel.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = gproc.decode(out[0][in_len:], skip_special_tokens=True)
    return text, parse_gemma_json(text, img.size)

raw, dets = gemma_detect(target_img)
print("Raw Gemma output:\n", raw)
print(f"\nParsed {len(dets)} detections.")

In [ ]:
# Visualise the Gemma JSON detections
if dets:
    boxes = [b for _, b in dets]; labels = [l for l, _ in dets]
    ax = show_boxes_pil(target_img, boxes, labels, color="#9333EA")
    ax.set_title("Gemma 3 — JSON-based open-vocabulary detection")
    plt.show()
else:
    print("No JSON parsed. Print `raw` above to see what the model returned.")

## 5. Three detectors, one verdict

| Aspect | YOLO11n | RT-DETR v2 (r18) | PaliGemma / Gemma 3 (VLM) |
|--------|---------|------------------|----------------------------|
| Architecture | CNN, anchor-free, decoupled head | Transformer, set prediction | ViT + LM, autoregressive |
| Output | bbox + class + score | bbox + class + score | text: `<loc>` tokens (PaliGemma) or **JSON** (Gemma 3) |
| NMS | yes | **no** (1-to-1 matching) | n/a (generated one at a time) |
| Open vocabulary | no | no | **yes** |
| Speed (T4) | ~100 fps | ~30 fps | ~1 generation / sec |
| Fine-tuning | seconds / minutes | minutes | tricky (LoRA/PEFT needed) |
| Where it wins | real-time, on-device, fixed classes | best of both, no NMS, transformer flexibility | zero-shot, natural-language conditioning |

> If you remember just one thing: **for the BAMBI project, start with YOLO11**. If you need *exotic* / unlabelled classes, *then* reach for a VLM.


## ✏️ Exercises

1. **Fine-tune YOLO on real data.** Replace the synthetic apple dataset with a small *real* set: take 20–30 photos of fruit on a table, label them in [Label Studio](https://labelstud.io) or [Roboflow](https://roboflow.com), export YOLO format, and re-train. How does mAP compare to the synthetic baseline?

2. **Match the same image with YOLO and RT-DETR.** For three of the supplied images, compute IoU between each pair of YOLO ↔ RT-DETR detections. Which classes do the two models *both* detect? Which does one find but the other misses? Plot a confusion-style matrix.

3. **Make the VLM count — and compare the two output formats.** Prompt PaliGemma (`<loc>` tokens) and Gemma 3 (JSON) to detect a countable class (e.g. apples) on the same 10 images. Compare each VLM's count to YOLO's. Which output format was easier to parse reliably, and where do the disagreements come from — missed detections, false positives, or malformed output?


In [ ]:
# Your code here